In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 04 · Baselines (naive, moving average, seasonal naive) on rolling-origin test windows

In [ ]:
import evaluate as ev
from app.features.feature_builder import make_features, CAL_KEYS
import json
z = np.load("data/processed/panel.npz"); meta = json.load(open("data/processed/series.json"))["series"]
cal = {k: z[f"cal_{k}"] for k in CAL_KEYS}; idx = {k: np.array([s[k] for s in meta]) for k in ["item_idx","store_idx","dept_idx","cat_idx"]}
D = z["sales"].shape[1] - 1; te = make_features(z["sales"], z["price"], cal, idx, [D - 28 * k for k in (4, 3, 2, 1)], with_target=True)
rows = {n: ev.all_metrics(te.y, te[c]) for n, c in {"naive_last_day": "lag1", "moving_avg_7": "roll_mean_7", "moving_avg_28": "roll_mean_28", "seasonal_naive": "snaive"}.items()}
pd.DataFrame(rows).T.round(3)